In [1]:
# 7월 24일 영화 제목만 soynlp로해서 매칭이 잘되도록 모델 학습 및 저장
# 1단계: WordExtractor 학습 및 tokenizer 저장
import json
import joblib
from soynlp.word import WordExtractor
from soynlp.tokenizer import LTokenizer

DB_PATH = "C:/kosmo/data/"
PATH = "C:/kosmo/test_data/IH/"
file_name = "movies_DB_partial.json"

# 영화 데이터 불러오기
with open(DB_PATH + file_name, encoding="utf-8") as f:
    movies = json.load(f)

# 학습용 문장 추출 (제목만 사용)
sentences = [movie.get('title_ko', '') for movie in movies]

# WordExtractor 학습
word_extractor = WordExtractor()
word_extractor.train(sentences)
word_scores = word_extractor.extract()

# 토크나이저 생성
tokenizer = LTokenizer(scores=word_scores)

# 저장
joblib.dump(word_scores, PATH + "soynlp_word_scores.pkl")
print("✅ WordExtractor 학습 및 tokenizer 저장 완료")

training was done. used memory 0.238 Gbory 0.202 Gb
all cohesion probabilities was computed. # words = 10005
all branching entropies was computed # words = 13386
all accessor variety was computed # words = 13386
✅ WordExtractor 학습 및 tokenizer 저장 완료


In [2]:
import json
import joblib
import numpy as np
from konlpy.tag import Okt
from gensim.models import Word2Vec
from soynlp.tokenizer import LTokenizer

# 경로 설정 (필요에 따라 수정)
DB_PATH = "C:/kosmo/data/"
PATH = "C:/kosmo/test_data/IH/"
file_name = "movies_DB_partial.json"

# 1) 영화 데이터 불러오기
with open(DB_PATH + file_name, encoding="utf-8") as f:
    movies = json.load(f)

# 2) soynlp word_scores 불러오기 (Scores 객체 → float 딕셔너리로 변환 필요)
raw_word_scores = joblib.load(PATH + "soynlp_word_scores.pkl")  # Scores 객체 딕셔너리 로드

# float 점수만 추출 (cohesion_forward 예시, 필요 시 다른 점수로 변경)
word_scores = {word: score.cohesion_forward for word, score in raw_word_scores.items()}

# 3) LTokenizer 생성
tokenizer = LTokenizer(scores=word_scores)

# 4) Okt 형태소 분석기 생성
okt = Okt()

# 5) 토큰화 함수 정의
def get_tokens(title, overview, genres, genre_weight=3):
    title = title or ""
    overview = overview or ""
    genres = genres or []

    tokens = []

    # 제목 토큰화: soynlp LTokenizer 사용
    title_tokens = tokenizer.tokenize(title)
    tokens.extend(title_tokens)

    # 줄거리 토큰화: konlpy Okt 사용 (명사, 동사, 형용사만)
    for word, pos in okt.pos(overview, stem=True, norm=True):
        if pos in ['Noun', 'Verb', 'Adjective']:
            tokens.append(word)

    # 장르 토큰 가중치 부여
    tokens.extend(genres * genre_weight)

    return tokens

# 6) 전체 영화 토큰화
keyword = []
for movie in movies:
    tokens = get_tokens(
        movie.get('title_ko'),
        movie.get('overview'),
        movie.get('genres'),
        genre_weight=3
    )
    keyword.append(tokens)

# 7) Word2Vec 모델 학습
model = Word2Vec(
    keyword,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0  # CBOW
)

# 8) 검색용 제목 토큰 저장 (줄거리/장르 제외)
title_tokens = [tokenizer.tokenize(movie.get('title_ko', '')) for movie in movies]
joblib.dump(title_tokens, PATH + "title_tokens.pkl")

# 9) 영화 벡터 생성 함수 정의
def get_movie_vector(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    return np.mean(np.array(vectors), axis=0) if vectors else np.zeros(model.vector_size)

# 10) 영화 벡터 계산
movie_vectors = [get_movie_vector(tokens, model) for tokens in keyword]

# 11) 모델 및 데이터 저장
joblib.dump(model, PATH + "word2vec_movie.model")
joblib.dump(movie_vectors, PATH + "movie_vectors.pkl")
joblib.dump(movies, PATH + "movies.pkl")
joblib.dump(keyword, PATH + "keyword.pkl")

print("✅ soynlp 기반 Word2Vec 학습 및 전체 데이터 저장 완료")

✅ soynlp 기반 Word2Vec 학습 및 전체 데이터 저장 완료


In [ ]:
import joblib
import json
import numpy as np
from soynlp.tokenizer import LTokenizer

# 현재 노트북 파일이 있는 디렉토리에 모델 및 데이터 파일이 있다고 가정

model = joblib.load("word2vec_movie.model")
movie_vectors = joblib.load("movie_vectors.pkl")
movies = joblib.load("movies.pkl")
title_tokens = joblib.load("title_tokens.pkl")
raw_word_scores = joblib.load("soynlp_word_scores.pkl")

word_scores = {w: s.cohesion_forward for w, s in raw_word_scores.items()}
tokenizer = LTokenizer(scores=word_scores)

# 제목 문자열 토큰화 함수
def tokenize_title(text):
    if not text:
        return []
    text = text.strip().replace(" ", "")
    return tokenizer.tokenize(text)

# 자카드 유사도 계산 함수
def jaccard_similarity(set1, set2):
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

# 입력한 제목과 가장 유사한 제목 영화 인덱스 반환
def find_similar_movie(input_title, title_tokens, threshold=0.1):
    input_tokens = set(tokenize_title(input_title))
    best_match_idx = -1
    best_score = 0.0
    for idx, tokens in enumerate(title_tokens):
        score = jaccard_similarity(input_tokens, set(tokens))
        if score > best_score and score >= threshold:
            best_score = score
            best_match_idx = idx
    return best_match_idx

# 영화 추천 함수
def recommend_movie(input_title, movies, movie_vectors, title_tokens, top_n=5):
    matched_idx = find_similar_movie(input_title, title_tokens)

    if matched_idx == -1:
        input_vec = np.mean(movie_vectors, axis=0)
    else:
        input_vec = movie_vectors[matched_idx]

    input_norm = np.linalg.norm(input_vec)
    vectors_norm = np.linalg.norm(movie_vectors, axis=1)
    dot_products = movie_vectors @ input_vec

    with np.errstate(divide='ignore', invalid='ignore'):
        cosine_similarities = dot_products / (vectors_norm * input_norm)
        cosine_similarities = np.nan_to_num(cosine_similarities)

    if matched_idx >= 0:
        cosine_similarities[matched_idx] = -1

    # 애니메이션 장르 페널티 적용
    adjusted_similarities = cosine_similarities.copy()
    for idx, m in enumerate(movies):
        if "애니메이션" in m.get("genres", []):
            adjusted_similarities[idx] *= 0.7

    top_indices = np.argpartition(-adjusted_similarities, top_n)[:top_n]
    top_indices = top_indices[np.argsort(-adjusted_similarities[top_indices])]

    recommendations = []
    for i in top_indices:
        m = movies[i]
        recommendations.append({
            "title_ko": m.get('title_ko', ''),
            "release_date": m.get('release_date', ''),
            "vote_average": m.get('vote_average', 0),
            "poster_path": m.get('poster_path', ''),
            "similarity": adjusted_similarities[i]
        })

    return recommendations

# ipynb에서 직접 실행할 수 있게 input_title 변수 정의
input_title = "취권"  # 원하는 검색어로 변경 가능

# 추천 실행 및 결과 출력
recs = recommend_movie(input_title, movies, movie_vectors, title_tokens, top_n=5)

result = {
    "input_title": input_title,
    "recommendations": []
}

# 추천 실행 및 결과 출력
recs = recommend_movie(input_title, movies, movie_vectors, title_tokens, top_n=5)

# 추천 영화별 유사도 출력
print(f"입력 영화: {input_title}")
print("추천 영화 및 유사도:")
for rec in recs:
    print(f"- {rec['title_ko']}: similarity = {rec['similarity']:.4f}")

result = {
    "input_title": input_title,
    "recommendations": []
}

for rec in recs:
    movie_info = next((m for m in movies if m.get('title_ko') == rec.get('title_ko')), {})
    result["recommendations"].append({
        "movie_id": movie_info.get("movie_id", ""),
        "title_ko": rec.get("title_ko", ""),
        "genres": movie_info.get("genres", []),
        "release_date": rec.get("release_date", ""),
        "vote_average": rec.get("vote_average", 0),
        "poster_path": rec.get("poster_path", "")
    })

print(json.dumps(result, ensure_ascii=False, indent=2))

입력 영화: 태극기휘날리며
추천 영화 및 유사도:
- 디오스: similarity = 0.9568
- 뚝방전설: similarity = 0.9486
- 드 러블리: similarity = 0.9475
- 더 챌린저: similarity = 0.9475
- 맨발의 꿈: similarity = 0.9470
{
  "input_title": "태극기휘날리며",
  "recommendations": [
    {
      "movie_id": 38874,
      "title_ko": "디오스",
      "genres": [
        "코미디"
      ],
      "release_date": "2001-11-30",
      "vote_average": 5.5,
      "poster_path": "https://image.tmdb.org/t/p/w500/gFgaMAdMfN4ZL1ouVeMd3XTIunQ.jpg"
    },
    {
      "movie_id": 40114,
      "title_ko": "뚝방전설",
      "genres": [
        "액션",
        "범죄"
      ],
      "release_date": "2006-09-07",
      "vote_average": 5.2,
      "poster_path": "https://image.tmdb.org/t/p/w500/tKFL9v8Yrk3ORH6DdRqUHJfWqNq.jpg"
    },
    {
      "movie_id": 15237,
      "title_ko": "드 러블리",
      "genres": [
        "드라마",
        "음악"
      ],
      "release_date": "2004-07-02",
      "vote_average": 6.152,
      "poster_path": "https://image.tmdb.org/t/p/w500/QBqiBQW7MGqlDSyvid4fKzS